# Train the student: baseline, QLoRA, evaluation, GGUF

Steps 5 to 8 of the guide, in one notebook. Run it with **Runtime → Change runtime type → T4 GPU**.

**Credentials:** `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` (the `pt-bio-extract-colab` user) and `DATA_BUCKET`.
In the Colab website, cell 1 reads them from **Secrets** (key icon, on the left). From VS Code or Antigravity,
Secrets can't be read, so cell 1 asks for each one in a password box.

**If Colab disconnects**, run cell 1 again and continue from where you were. Every result is copied to
Google Drive and S3 as soon as it is ready, and the cells skip work that is already done:
the baseline, the training (it resumes from the last checkpoint) and the student's predictions.

## 1. Set up

In [ ]:
VERSION = "v1"  # the same as in 01_label_teacher.ipynb

!pip install -q -U transformers trl peft bitsandbytes accelerate datasets boto3 pydantic awscli
# Colab ships torchao; a version that does not match peft makes PeftModel.from_pretrained
# fail with an ImportError. This project does not use it
!pip uninstall -q -y torchao

import gc, glob, json, os
import torch
from google.colab import drive, userdata

from getpass import getpass

def secret(name):
    """Colab Secrets in the browser; outside it (VS Code, Antigravity) they can't be read,
    so ask for the value instead. It stays in this session's memory only."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        return userdata.get(name).strip()  # a stray space in a secret breaks the AWS credentials
    except Exception:  # TimeoutException outside the browser, or a missing secret
        return getpass(f"{name}: ").strip()

for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "DATA_BUCKET"):
    os.environ[k] = secret(k)
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"
# Also a Python variable: shell lines below use {DATA_BUCKET}, never $DATA_BUCKET mixed with {...}
DATA_BUCKET = os.environ["DATA_BUCKET"]

drive.mount("/content/drive")  # anything slow goes to Drive: it survives disconnects
DRIVE = f"/content/drive/MyDrive/wiki-bio-distill/{VERSION}"
RUN_DIR = f"{DRIVE}/run"
os.makedirs(f"{DRIVE}/preds", exist_ok=True)

# The repository is public: no token needed
url = "https://github.com/pinthoz/wiki-bio-distill.git"
if not os.path.exists("/content/repo"):
    !git clone -q {url} /content/repo
%cd /content/repo
!git pull -q
!mkdir -p data preds

BASE = "Qwen/Qwen2.5-1.5B-Instruct"

def save(path, s3_name=None):
    """Copy a result to Drive and S3 right away: Colab's disk is wiped when the session ends."""
    !cp {path} {DRIVE}/{path}
    !aws s3 cp {path} s3://{DATA_BUCKET}/{VERSION}/{s3_name or path} --only-show-errors
    print(f"saved {path}")

def free_gpu(*names):
    for n in names:
        globals().pop(n, None)
    gc.collect(); torch.cuda.empty_cache()

!aws sts get-caller-identity --query Arn --output text   # must end in user/pt-bio-extract-colab
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
for f in ("train", "val", "test"):
    !aws s3 cp s3://{DATA_BUCKET}/{VERSION}/{f}.jsonl data/ --only-show-errors
!wc -l data/*.jsonl

## 2. Baseline: the base model without training, with the detailed prompt

The base model gets the teacher's full prompt: it is the best it can do without fine-tuning.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from common import SYSTEM_TEACHER
from gen import generate_preds

test = [json.loads(l) for l in open("data/test.jsonl", encoding="utf-8")]

if os.path.exists(f"{DRIVE}/preds/base.jsonl"):  # done in an earlier session
    !cp {DRIVE}/preds/base.jsonl preds/
else:
    tokenizer = AutoTokenizer.from_pretrained(BASE)
    base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto")
    generate_preds(base, tokenizer, test, SYSTEM_TEACHER, "preds/base.jsonl")
    free_gpu("base")
    save("preds/base.jsonl")

!python evaluate.py preds/base.jsonl
!head -n 3 preds/base.jsonl

## 3. QLoRA fine-tuning (about 30–60 min on a T4)

Skipped if the adapter is already on Drive. If Colab disconnected mid-run, it resumes from the last checkpoint.

In [ ]:
from datasets import load_dataset
from peft import LoraConfig
from transformers import BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

def train():
    tokenizer = AutoTokenizer.from_pretrained(BASE)
    tokenizer.padding_side = "right"  # gen.py sets "left" for generation; training uses right
    model = AutoModelForCausalLM.from_pretrained(
        BASE,
        dtype=torch.float16,  # the T4 has no bf16: keep the layers that are not quantized in fp16
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",            # 4-bit type made for neural network weights
            bnb_4bit_use_double_quant=True,       # also quantizes the scale constants
            bnb_4bit_compute_dtype=torch.float16,
        ),
        device_map="auto",
    )
    data = load_dataset("json", data_files={"train": "data/train.jsonl", "validation": "data/val.jsonl"})
    peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                             target_modules="all-linear", task_type="CAUSAL_LM")
    args = SFTConfig(
        output_dir=RUN_DIR, num_train_epochs=2,
        per_device_train_batch_size=8, gradient_accumulation_steps=2,  # effective batch = 16
        learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=10,
        max_length=1024, fp16=True, gradient_checkpointing=True,
        logging_steps=10, eval_strategy="steps", eval_steps=50, save_steps=50, save_total_limit=3,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", report_to="none",
    )
    trainer = SFTTrainer(model=model, args=args, train_dataset=data["train"],
                         eval_dataset=data["validation"], processing_class=tokenizer,
                         peft_config=peft_config)
    # LoRA weights in fp32: fp16 mixed precision cannot unscale fp16/bf16 gradients
    for p in trainer.model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    trainer.model.print_trainable_parameters()

    resume = bool(glob.glob(f"{RUN_DIR}/checkpoint-*"))
    trainer.train(resume_from_checkpoint=resume)
    trainer.save_model(f"{RUN_DIR}/adapter")
    tokenizer.save_pretrained(f"{RUN_DIR}/adapter")
    print("final eval loss:", trainer.evaluate()["eval_loss"])

if os.path.exists(f"{RUN_DIR}/adapter/adapter_model.safetensors"):
    print("adapter already on Drive: skipping training")
else:
    train()
    free_gpu()

!aws s3 cp --recursive {RUN_DIR}/adapter s3://{DATA_BUCKET}/{VERSION}/models/qwen1.5b-lora-r16/adapter/ --only-show-errors

## 4. Evaluate the student (adapter merged into the fp16 model)

In [ ]:
from peft import PeftModel
from common import SYSTEM_STUDENT

def load_student():
    tokenizer = AutoTokenizer.from_pretrained(f"{RUN_DIR}/adapter")
    base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto")
    return PeftModel.from_pretrained(base, f"{RUN_DIR}/adapter").merge_and_unload(), tokenizer

if os.path.exists(f"{DRIVE}/preds/student.jsonl"):  # done in an earlier session
    !cp {DRIVE}/preds/student.jsonl preds/
else:
    student, tokenizer = load_student()
    generate_preds(student, tokenizer, test, SYSTEM_STUDENT, "preds/student.jsonl")
    save("preds/student.jsonl")

!python evaluate.py preds/base.jsonl preds/student.jsonl
save("results.json", "results-hf.json")

## 5. Error analysis

For each error: did the teacher get it wrong too? Then the problem is in the labels. If only the student did, it needs more examples of that kind or more capacity.

In [ ]:
from common import FIELDS, as_set, parse_json

preds = {p["id"]: parse_json(p["raw"]) for p in map(json.loads, open("preds/student.jsonl", encoding="utf-8"))}
shown = 0
for t in test:
    p = preds[t["id"]] or {}
    wrong = [f for f in FIELDS if as_set(p.get(f)) != as_set(t["gold"].get(f))]
    if wrong and shown < 10:
        shown += 1
        print(t["text"][:200], "…")
        for f in wrong:
            print(f"  {f}: gold={t['gold'].get(f)!r}  student={p.get(f)!r}  teacher={t['teacher'].get(f)!r}")
        print()

## 6. Export to GGUF and quantize to Q4_K_M

Only llama.cpp's `gguf` package is installed. **Do not** run llama.cpp's `requirements.txt` here: it installs CPU PyTorch and an old transformers.

In [ ]:
if "student" not in globals():
    student, tokenizer = load_student()

MERGED = "/content/merged"
student.save_pretrained(MERGED)
tokenizer.save_pretrained(MERGED)
free_gpu("student", "base")

!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q /content/llama.cpp/gguf-py sentencepiece

!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED} --outtype f16 --outfile /content/student-f16.gguf
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j > /dev/null
!/content/llama.cpp/build/bin/llama-quantize /content/student-f16.gguf /content/student-q4_k_m.gguf Q4_K_M
!cd /content && sha256sum student-q4_k_m.gguf > student-q4_k_m.gguf.sha256
!ls -lh /content/*.gguf

GGUF_S3 = f"s3://{DATA_BUCKET}/{VERSION}/models/qwen1.5b-lora-r16"
!cp /content/student-q4_k_m.gguf /content/student-q4_k_m.gguf.sha256 {DRIVE}/
!aws s3 cp /content/student-q4_k_m.gguf {GGUF_S3}/ --only-show-errors
!aws s3 cp /content/student-q4_k_m.gguf.sha256 {GGUF_S3}/ --only-show-errors

## 7. Quantization loss on CPU, with a JSON grammar as in the Lambda

In [ ]:
!pip install -q llama-cpp-python==0.3.35   # builds on CPU: a few minutes

import time
from llama_cpp import Llama
from common import SCHEMA, messages

llm = Llama(model_path="/content/student-q4_k_m.gguf", n_ctx=2048, verbose=False)
with open("preds/student-q4.jsonl", "w", encoding="utf-8") as f:
    for t in test:
        t0 = time.perf_counter()
        out = llm.create_chat_completion(
            messages=messages(t["text"], SYSTEM_STUDENT), temperature=0, max_tokens=300,
            response_format={"type": "json_object", "schema": SCHEMA})
        f.write(json.dumps({"id": t["id"], "raw": out["choices"][0]["message"]["content"],
                            "ms": round((time.perf_counter() - t0) * 1000)}, ensure_ascii=False) + "\n")
save("preds/student-q4.jsonl")

!python evaluate.py preds/base.jsonl preds/student.jsonl preds/student-q4.jsonl
save("results.json", "results-hf.json")

Done. Everything is in `s3://<DATA_BUCKET>/<VERSION>/`: `preds/`, `results-hf.json`, and the adapter and the GGUF in `models/qwen1.5b-lora-r16/`.
On your laptop, `aws s3 cp s3://$DATA_BUCKET/v1/preds/ preds/ --recursive` (or your `VERSION`) brings the predictions back for the full comparison table.